# 06. Application + POS/CASH

## Цель

Кратко:
- создать признаки из `POS_CASH_balance.csv`;
- добавить только POS/CASH-признаки к `application_train`;
- обучить CatBoost и записать `application_pos_cash`.


## 1. Импорты и пути


In [1]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

Mounted at /content/drive
Installing missing dependency: catboost
Environment: Google Colab
Python: 3.12.13
Project root: /content/drive/MyDrive/credit-scoring-system
Raw data: /content/drive/MyDrive/credit-scoring-system/data/raw
Models: /content/drive/MyDrive/credit-scoring-system/models
Reports: /content/drive/MyDrive/credit-scoring-system/reports


In [2]:
import numpy as np
import pandas as pd
from catboost import (
    CatBoostClassifier,
    Pool,
    cv as catboost_cv,
)
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

from src.config import (
    INTERIM_DATA_DIR as DATA_INTERIM_DIR,
    PROCESSED_DATA_DIR as DATA_PROCESSED_DIR,
    find_data_file,
)
from src.experiment_tracking import save_experiment_result
from src.model_config import (
    get_catboost_device_config,
    get_catboost_gpu_count,
    print_catboost_device_info,
)


APPLICATION_PATH = find_data_file("application_train.csv")
POS_CASH_PATH = find_data_file("POS_CASH_balance.csv")
CLIENT_SPLIT_PATH = (
    DATA_PROCESSED_DIR / "client_split.csv"
)
POS_CASH_FEATURES_PATH = DATA_INTERIM_DIR / "pos_cash_features.csv"
RANDOM_STATE = 42
CV_FOLDS = 3


### Устройство CatBoost


In [3]:
gpu_count = get_catboost_gpu_count()
catboost_device_config = get_catboost_device_config(
    gpu_count=gpu_count,
)
print_catboost_device_info(
    catboost_device_config,
    gpu_count=gpu_count,
)


Modeling environment: Google Colab
CatBoost GPU count: 1
CatBoost device: GPU
CatBoost GPU devices: 0


## 2. Загрузка application_train

Основная таблица содержит одну строку на клиента. Техническое значение
`365243` в `DAYS_EMPLOYED` заменяется пропуском.


In [4]:
application = pd.read_csv(APPLICATION_PATH)

if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application[
        "DAYS_EMPLOYED"
    ].replace(365243, np.nan)

assert application["SK_ID_CURR"].is_unique
assert application["TARGET"].isin([0, 1]).all()

print("Application:", application.shape)
display(application.head())


Application: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Загрузка POS_CASH_balance


In [5]:
pos_cash = pd.read_csv(POS_CASH_PATH)

assert {"SK_ID_CURR", "SK_ID_PREV"}.issubset(
    pos_cash.columns
)

print("POS/CASH:", pos_cash.shape)
display(pos_cash.head())


POS/CASH: (10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


## 4. Признаки на уровне месячной записи

Используются только доступные к моменту заявки месяцы. Отдельно отмечаем
просрочку и активный кредит.


In [6]:
pos_cash = pos_cash[
    pos_cash["MONTHS_BALANCE"].le(0)
].copy()

pos_cash["HAS_DELINQUENCY"] = (
    pos_cash["SK_DPD"].gt(0)
).astype(int)

pos_cash["ACTIVE_CREDIT_ID"] = pos_cash[
    "SK_ID_PREV"
].where(
    pos_cash["NAME_CONTRACT_STATUS"].eq("Active")
)


## 5. Агрегация до клиента


In [7]:
pos_cash_features = (
    pos_cash
    .groupby("SK_ID_CURR")
    .agg(
        POS_RECORD_COUNT=("MONTHS_BALANCE", "count"),
        POS_DPD_MEAN=("SK_DPD", "mean"),
        POS_DPD_MAX=("SK_DPD", "max"),
        POS_DELINQUENCY_SHARE=("HAS_DELINQUENCY", "mean"),
        POS_REMAINING_INSTALMENTS_MEAN=(
            "CNT_INSTALMENT_FUTURE",
            "mean",
        ),
        POS_ACTIVE_CREDIT_COUNT=("ACTIVE_CREDIT_ID", "nunique"),
        POS_LAST_OBSERVATION_MONTH=("MONTHS_BALANCE", "max"),
    )
    .reset_index()
)

pos_cash_features[
    "POS_LAST_OBSERVATION_MONTH"
] *= -1

assert pos_cash_features["SK_ID_CURR"].is_unique

print(pos_cash_features.shape)
display(pos_cash_features.head())


(337252, 8)


,SK_ID_CURR,POS_RECORD_COUNT,POS_DPD_MEAN,POS_DPD_MAX,POS_DELINQUENCY_SHARE,POS_REMAINING_INSTALMENTS_MEAN,POS_ACTIVE_CREDIT_COUNT,POS_LAST_OBSERVATION_MONTH
0,100001,9,0.777778,7,0.111111,1.444444,2,53
1,100002,19,0.000000,0,0.000000,15.000000,1,1
2,100003,28,0.000000,0,0.000000,5.785714,3,18
3,100004,4,0.000000,0,0.000000,2.250000,1,24
4,100005,11,0.000000,0,0.000000,7.200000,1,15


## 6. Проверка и сохранение признаков


In [8]:
assert pos_cash_features["SK_ID_CURR"].is_unique
assert "TARGET" not in pos_cash_features.columns

pos_cash_features.to_csv(
    POS_CASH_FEATURES_PATH,
    index=False,
)

print("Сохранено:", POS_CASH_FEATURES_PATH)


Сохранено: /content/drive/MyDrive/credit-scoring-system/data/interim/pos_cash_features.csv


## 7. Merge с application


In [9]:
modeling_data = application.merge(
    pos_cash_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print("Application:", application.shape)
print("После добавления POS/CASH:", modeling_data.shape)
print(
    "Добавлено признаков:",
    modeling_data.shape[1] - application.shape[1],
)


Application: (307511, 122)
После добавления POS/CASH: (307511, 129)
Добавлено признаков: 7


## Чтение единого client split


In [10]:
if not CLIENT_SPLIT_PATH.exists():
    raise FileNotFoundError(
        "Сначала выполните notebooks/02_application_baseline.ipynb. "
        f"Ожидаемый файл: {CLIENT_SPLIT_PATH}"
    )

client_split = pd.read_csv(CLIENT_SPLIT_PATH)

assert client_split.columns.tolist() == ["SK_ID_CURR", "split"]
assert client_split["SK_ID_CURR"].is_unique
assert set(client_split["split"]) == {"train", "holdout"}
assert set(client_split["SK_ID_CURR"]) == set(application["SK_ID_CURR"])

modeling_data = modeling_data.merge(
    client_split,
    on="SK_ID_CURR",
    how="inner",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print(client_split["split"].value_counts())


split
train      246008
holdout     61503
Name: count, dtype: int64


## 9. Создание X и y

`TARGET`, идентификатор клиента и техническая колонка разделения не
передаются модели.


In [11]:
train_data = modeling_data[
    modeling_data["split"].eq("train")
].copy()

n_holdout = int(
    modeling_data["split"].eq("holdout").sum()
)

feature_columns = [
    column
    for column in modeling_data.columns
    if column not in {
        "TARGET",
        "SK_ID_CURR",
        "split",
    }
]

X_train = train_data[feature_columns]
y_train = train_data["TARGET"].astype(int)

assert "TARGET" not in X_train.columns
assert "SK_ID_CURR" not in X_train.columns
assert "split" not in X_train.columns

print("Train:", X_train.shape)
print("Holdout clients (не используется):", n_holdout)


Train: (246008, 127)
Holdout clients (не используется): 61503


## 10. Подготовка данных для CatBoost

Категориальные пропуски заменяются строкой. Числовые `NaN` остаются без
изменений: CatBoost обрабатывает их самостоятельно.


In [12]:
categorical_columns = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

X_train_catboost = X_train.copy()
X_train_catboost[categorical_columns] = (
    X_train_catboost[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

print("Категориальных признаков:", len(categorical_columns))


Категориальных признаков: 16


## 11. Стратифицированная кросс-валидация


In [13]:
cv_splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


## 12. Pool для CatBoost


In [14]:
train_pool = Pool(
    data=X_train_catboost,
    label=y_train,
    cat_features=categorical_columns,
)


## 13. Параметры CatBoost


In [15]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "custom_metric": ["PRAUC:type=Classic"],
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": False,
    **catboost_device_config,
}


## 14. Библиотечная CV CatBoost

OOF-предсказания не требуются, поэтому используется `catboost.cv()`
без ручного цикла по фолдам. Test в CV не участвует.


In [16]:
catboost_cv_results = catboost_cv(
    pool=train_pool,
    params=catboost_params,
    folds=cv_splitter,
    early_stopping_rounds=100,
    as_pandas=True,
    verbose=100,
)


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:877: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(
Default metric period is 5 because AUC, PRAUC is/are not implemented for GPU


Training on fold [0/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.6973384	best: 0.6973384 (0)	total: 172ms	remaining: 2m 51s
100:	test: 0.7476591	best: 0.7476593 (99)	total: 10.3s	remaining: 1m 31s
200:	test: 0.7501743	best: 0.7501743 (200)	total: 19.1s	remaining: 1m 16s
300:	test: 0.7518162	best: 0.7518162 (299)	total: 28.2s	remaining: 1m 5s
400:	test: 0.7544720	best: 0.7544720 (397)	total: 38.4s	remaining: 57.4s
500:	test: 0.7561613	best: 0.7561697 (496)	total: 46.5s	remaining: 46.3s
600:	test: 0.7572106	best: 0.7572294 (598)	total: 56.2s	remaining: 37.3s
700:	test: 0.7578041	best: 0.7578041 (700)	total: 1m 6s	remaining: 28.3s
800:	test: 0.7585513	best: 0.7585513 (800)	total: 1m 14s	remaining: 18.5s
900:	test: 0.7590704	best: 0.7590704 (899)	total: 1m 24s	remaining: 9.25s
999:	test: 0.7595096	best: 0.7595258 (995)	total: 1m 34s	remaining: 0us
bestTest = 0.7595258355
bestIteration = 995
Training on fold [1/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7043935	best: 0.7043935 (0)	total: 162ms	remaining: 2m 41s
100:	test: 0.7502226	best: 0.7502226 (100)	total: 9.5s	remaining: 1m 24s
200:	test: 0.7542337	best: 0.7542337 (200)	total: 18.5s	remaining: 1m 13s
300:	test: 0.7576317	best: 0.7576666 (295)	total: 28.3s	remaining: 1m 5s
400:	test: 0.7589114	best: 0.7589114 (397)	total: 36.3s	remaining: 54.2s
500:	test: 0.7598218	best: 0.7598218 (495)	total: 45.8s	remaining: 45.7s
600:	test: 0.7604094	best: 0.7604094 (600)	total: 55.5s	remaining: 36.8s
700:	test: 0.7612187	best: 0.7612189 (687)	total: 1m 3s	remaining: 27.1s
800:	test: 0.7619177	best: 0.7619195 (799)	total: 1m 13s	remaining: 18.3s
900:	test: 0.7627055	best: 0.7627065 (895)	total: 1m 23s	remaining: 9.16s
999:	test: 0.7634313	best: 0.7634336 (997)	total: 1m 31s	remaining: 0us
bestTest = 0.7634336352
bestIteration = 997
Training on fold [2/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7060657	best: 0.7060657 (0)	total: 224ms	remaining: 3m 43s
100:	test: 0.7499053	best: 0.7499053 (100)	total: 9.87s	remaining: 1m 27s
200:	test: 0.7527741	best: 0.7527741 (199)	total: 19.7s	remaining: 1m 18s
300:	test: 0.7542304	best: 0.7542304 (296)	total: 27.6s	remaining: 1m 4s
400:	test: 0.7567117	best: 0.7567117 (400)	total: 37.3s	remaining: 55.7s
500:	test: 0.7581388	best: 0.7581631 (482)	total: 47.1s	remaining: 46.9s
600:	test: 0.7592996	best: 0.7593036 (596)	total: 55.3s	remaining: 36.7s
700:	test: 0.7597821	best: 0.7597821 (698)	total: 1m 5s	remaining: 27.7s
800:	test: 0.7600950	best: 0.7600952 (788)	total: 1m 14s	remaining: 18.4s
900:	test: 0.7602981	best: 0.7602981 (900)	total: 1m 22s	remaining: 9.1s
999:	test: 0.7608057	best: 0.7608150 (990)	total: 1m 32s	remaining: 0us
bestTest = 0.7608149648
bestIteration = 990


## 15. Лучшая итерация и CV-метрики


In [17]:
auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-mean")
)

auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-std")
)

pr_auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-mean")
)

pr_auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-std")
)

best_cv_index = catboost_cv_results[
    auc_mean_column
].idxmax()

best_cv_row = catboost_cv_results.loc[
    best_cv_index
]

best_iteration = int(
    best_cv_row["iterations"]
) + 1

cv_roc_auc = float(
    best_cv_row[auc_mean_column]
)

cv_roc_auc_std = float(
    best_cv_row[auc_std_column]
)

cv_pr_auc = float(
    best_cv_row[pr_auc_mean_column]
)

cv_pr_auc_std = float(
    best_cv_row[pr_auc_std_column]
)

print(f"Лучшая итерация: {best_iteration}")
print(
    f"CV ROC-AUC: {cv_roc_auc:.4f} "
    f"± {cv_roc_auc_std:.4f}"
)
print(
    f"CV PR-AUC: {cv_pr_auc:.4f} "
    f"± {cv_pr_auc_std:.4f}"
)


Лучшая итерация: 996
CV ROC-AUC: 0.7612 ± 0.0020
CV PR-AUC: nan ± nan


## 16. Итоговая модель CatBoost


In [18]:
catboost_model = CatBoostClassifier(
    iterations=best_iteration,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
    **catboost_device_config,
)


## 17. Обучение итоговой модели


In [19]:
catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_columns,
)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 62.7ms	remaining: 1m 2s
100:	total: 4.84s	remaining: 42.9s
200:	total: 10.1s	remaining: 40.1s
300:	total: 14.4s	remaining: 33.2s
400:	total: 19s	remaining: 28.2s
500:	total: 24.4s	remaining: 24.1s
600:	total: 28.6s	remaining: 18.8s
700:	total: 33.2s	remaining: 14s
800:	total: 38.5s	remaining: 9.37s
900:	total: 42.7s	remaining: 4.5s
995:	total: 46.9s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, devices='0', eval_metric='AUC', iterations=996, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

## Запись CV-результата


In [20]:
current_result = {
    "experiment": "application_pos_cash",
    "notebook": "06_pos_cash_features.ipynb",
    "model": "CatBoostClassifier",
    "feature_set": "application + POS_CASH",
    "source_tables": "application_train.csv, POS_CASH_balance.csv",
    "device": catboost_device_config["task_type"],
    "n_train": len(train_data),
    "n_holdout": n_holdout,
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": best_iteration,
    "cv_roc_auc": cv_roc_auc,
    "cv_roc_auc_std": cv_roc_auc_std,
    "cv_pr_auc": cv_pr_auc,
    "cv_pr_auc_std": cv_pr_auc_std,
    "holdout_roc_auc": None,
    "holdout_pr_auc": None,
}

all_results = save_experiment_result(current_result)
display(all_results)


,experiment,notebook,model,feature_set,source_tables,device,n_train,n_holdout,n_features,cv_folds,best_iteration,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,cv_pr_auc_std,holdout_roc_auc,holdout_pr_auc
0,application_logistic,02_application_baseline.ipynb,LogisticRegression,application,application_train.csv,CPU,246008,61503,120,3,NaN,0.744841,0.002434,0.217865,0.005206,NaN,NaN
1,application_catboost,02_application_baseline.ipynb,CatBoostClassifier,application,application_train.csv,GPU,246008,61503,120,3,1000.0,0.754176,0.002317,NaN,NaN,NaN,NaN
2,application_bureau,03_bureau_features.ipynb,CatBoostClassifier,application + bureau,"application_train.csv, bureau.csv, bureau_bala...",GPU,246008,61503,134,3,1000.0,0.758362,0.001620,NaN,NaN,NaN,NaN
3,application_previous,04_previous_application_features.ipynb,CatBoostClassifier,application + previous_application,"application_train.csv, previous_application.csv",GPU,246008,61503,131,3,1000.0,0.760286,0.002017,NaN,NaN,NaN,NaN
4,application_installments,05_installments_features.ipynb,CatBoostClassifier,application + installments,"application_train.csv, installments_payments.csv",GPU,246008,61503,129,3,1000.0,0.759957,0.000499,NaN,NaN,NaN,NaN
5,application_pos_cash,06_pos_cash_features.ipynb,CatBoostClassifier,application + POS_CASH,"application_train.csv, POS_CASH_balance.csv",GPU,246008,61503,127,3,996.0,0.761249,0.001983,NaN,NaN,NaN,NaN


## Выводы


In [21]:
print("Эксперимент: application + POS_CASH_balance")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Лучшая итерация: {best_iteration}")
print(f"CV ROC-AUC: {cv_roc_auc:.4f}")
print(f"CV PR-AUC: {cv_pr_auc:.4f}")
print("Holdout не использовался: результат сравнивается только по CV.")


Эксперимент: application + POS_CASH_balance
Количество признаков: 127
Лучшая итерация: 996
CV ROC-AUC: 0.7612
CV PR-AUC: nan
Holdout не использовался: результат сравнивается только по CV.
